# PSF extraction from bead images

Detects beads in all TIFFs of a dataset folder, refines them with a Gaussian fit,
applies QC (nearest-neighbour distance + MAD on per-bead FWHM), and averages the
subvoxel-aligned bead crops into a PSF - **per input file and pooled over all
files** (`ALL_*`). Optionally distills the finite bead structure out of the PSF.

- Inputs: `in_path/<tif_subdir>/*.tif` (ImageJ TIFFs with pixel size, or set `psz`)
- Outputs in `in_path/<out_subdir>/`: `ALL_psf_avg.tif` + `ALL_psf_distilled.tif`
  with JSON sidecars (psz, `psf_metrics`, provenance), per-file `<stem>_psf_*.tif`
  + sidecars, pooled `detections.csv` (with QC mask columns) and per-file
  `detections_<stem>.csv`, optional `aligned_stack.npz`, report figures
  `report_ALL.png` + one `report_<stem>.png` per input file (orthogonal
  center cuts + marginals with FWHM markers; `_distilled` variants when
  distilling), `recipe_parameters.json`. If the input files have different
  pixel sizes, a warning is issued and only the per-file PSFs/reports are
  produced (each with its own psz) - no `ALL_*` pooled outputs.

Generic functions live in `calmutils.restoration` / `calmutils.morphology`; the
bead workflow layer is the sibling module `psf_utils.py` (self-test:
`python psf_utils.py`).

Run interactively, or via papermill / `recipe_runner.py` (parameters cell is
tagged `parameters`; the runner can skip finished datasets with
`done_if_exists: "{in_path}/psf/ALL_psf_avg.tif"`; with unequal pixel sizes no
`ALL_*` is written, so use a per-file tif or `detections.csv` as the marker).

In [ ]:
# dataset folder containing the bead tiffs
in_path = '/Users/david/Desktop/scratch_data/20230616_SpinningDiskPSF'

# subfolder with tiffs (relative to in_path), '' if directly in in_path
tif_subdir = 'tif'

# results subfolder (relative to in_path)
out_subdir = 'psf'

# voxel size [z, y, x] in nm; None = read from ImageJ tiff metadata
psz = None

# PSF crop window in voxels, odd per axis (well-defined center voxel)
psf_size = [13, 29, 29]

# expected bead size in nm (z, y, x) - only seeds the LoG detection
expected_size_nm = [500, 250, 250]
blob_threshold_rel = 0.15

# QC: reject beads with a too-close nearest neighbour (PSF-wing contamination).
# absolute threshold in nm (preferred: identical criterion across datasets) ...
min_nn_distance_nm = None
# ... and/or reject the lowest quantile of NN distances (self-adaptive).
# if both are set, the larger threshold wins
nn_quantile = 0.1
# absolute starting point instead: the crop window extent max(psf_size) * psz
# QC: MAD outlier rejection on per-bead mean xy FWHM ('high' rejects aggregates)
n_mad = 4.0
mad_side = 'high'

# PSF distillation: deconvolve the finite bead structure (solid sphere model)
distill = True
bead_diameter_nm = 170.0
num_iter = 25
# per-axis OTF cutoff in cycles per nm (e.g. 2*NA/lambda_em laterally); None = off
bandlimit = None

edge = 'skip'            # 'skip' | 'partial' crops of beads at the image border
save_aligned_stack = True
fwhm_outlier_rel = 0.2   # warn if a per-file PSF FWHM deviates from pooled by more than this

# report figures: gamma applied to the imshow cuts (normalized to the PSF peak);
# <1 brightens dim wings/sidelobes, 1 = linear. Marginals stay linear (FWHM markers).
report_gamma = 0.5

In [ ]:
# << auto-managed by scripts/inject_parameter_logging.py - do not edit >>
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'recipe_params.py').is_file():
        sys.path.insert(0, str(_p))
        break
import recipe_params
recipe_params.capture(globals(), notebook='restoration/psf_extraction.ipynb')

In [ ]:
import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from matplotlib import pyplot as plt
from skimage.feature import blob_log

from calmutils.imageio.tiff_imagej import save_tiff_imagej, get_imagej_tiff_pixel_size
from calmutils.localization import refine_point_lsq
from calmutils.localization.util import sig_to_full_width_at_quantile, nearest_neighbor_distances
from calmutils.restoration import psf_metrics

# sibling module psf_utils.py: search cwd and its parents (works for
# interactive runs from restoration/ and papermill runs from the repo root)
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'restoration' / 'psf_utils.py').is_file():
        sys.path.insert(0, str(_p / 'restoration'))
        break
from psf_utils import (align_bead_stack, average_stack, distill_psf,
                       filter_close_beads, per_bead_metrics, filter_mad_outliers)

in_path = Path(in_path)
tif_dir = in_path / tif_subdir if tif_subdir else in_path
out_path = in_path / out_subdir
out_path.mkdir(parents=True, exist_ok=True)
print(f'input tiffs : {tif_dir}')
print(f'output      : {out_path}')

In [ ]:
tif_files = sorted(set(tif_dir.glob('*.tif')) | set(tif_dir.glob('*.tiff')))
if not tif_files:
    raise FileNotFoundError(f'no .tif/.tiff files in {tif_dir}')

if psz is None:
    sizes, units = zip(*[get_imagej_tiff_pixel_size(f) for f in tif_files])
    unit = units[0]
    to_nm = {'nm': 1.0, 'um': 1e3, 'µm': 1e3, 'micron': 1e3, 'mm': 1e6, 'cm': 1e7, 'm': 1e9}.get(unit)
    if to_nm is None:
        raise ValueError(f'cannot interpret pixel unit {unit!r} - set psz explicitly')
    psz_per_file = [np.asarray(s, dtype=float) * to_nm for s in sizes]
else:
    # an explicit psz applies to all files
    psz_per_file = [np.asarray(psz, dtype=float)] * len(tif_files)

psz_arr = psz_per_file[0]
uniform_psz = all(np.allclose(p, psz_arr) for p in psz_per_file[1:])
if not uniform_psz:
    warnings.warn("pixel size differs between input files - per-file PSFs will use each "
                  "file's own pixel size and no pooled ALL_* PSF is written "
                  "(set psz explicitly to force a common pixel size)")

print(f'{len(tif_files)} tiff(s): {[f.name for f in tif_files]}')
for f, p in zip(tif_files, psz_per_file):
    print(f'  {f.name}: psz (nm) z,y,x = {p}')


In [ ]:
def detect_beads(img, expected_size_nm, psz_nm, threshold_rel):
    # LoG detection seeded at the expected bead size, then least-squares Gaussian fit
    sigma_expected = (np.asarray(expected_size_nm) / psz_nm
                      / sig_to_full_width_at_quantile(np.ones(img.ndim)))
    blobs = blob_log(img, min_sigma=sigma_expected, num_sigma=1,
                     threshold=None, threshold_rel=threshold_rel)
    cols = ['pos_z', 'pos_y', 'pos_x', 'sigma_z', 'sigma_y', 'sigma_x', 'peak']
    rows = []
    for blob in blobs:
        pos, fit = refine_point_lsq(img, blob[:3])
        if fit is None:
            continue
        fit = fit[0]  # [bg, amplitude, z, y, x, sigma_z, sigma_y, sigma_x]
        if np.any(fit[-3:] < 0):
            continue
        rows.append(list(pos) + list(fit[-3:]) + [fit[1]])
    return pd.DataFrame(rows, columns=cols)

frames = []
for fi, f in enumerate(tif_files):
    img = tifffile.imread(f).astype(float)
    assert img.ndim == 3, f'{f.name}: expected a 3D stack (z, y, x), got {img.shape}'
    det = detect_beads(img, expected_size_nm, psz_per_file[fi], blob_threshold_rel)
    det['file'] = f.name
    det.to_csv(out_path / f'detections_{f.stem}.csv', index=False)
    frames.append(det)
    print(f'{f.name}: shape {img.shape}, {len(det)} beads detected')

detections = pd.concat(frames, ignore_index=True)
if len(detections) == 0:
    raise RuntimeError('no beads detected in any input file')
locs = detections[['pos_z', 'pos_y', 'pos_x']].to_numpy()

In [ ]:
psf_size_t = tuple(int(s) for s in psf_size)

# align all bead crops (per file, then pooled)
stacks, used_parts, file_idx = [], [], []
for fi, f in enumerate(tif_files):
    img = tifffile.imread(f).astype(float)
    locs_f = detections.loc[detections.file == f.name,
                            ['pos_z', 'pos_y', 'pos_x']].to_numpy()
    stack_f, used_f = align_bead_stack(img, locs_f, psf_size_t, edge=edge)
    stacks.append(stack_f)
    used_parts.append(used_f)
    file_idx.append(np.full(len(locs_f), fi, dtype=int))

stack = np.concatenate(stacks)
used = np.concatenate(used_parts)
file_idx = np.concatenate(file_idx)

# per-bead metrics + QC masks; NN distances are computed per file (positions of
# different files are unrelated, and each file has its own pixel size)
m = per_bead_metrics(stack, used, psz=psz_arr)  # only fwhm_px (voxel units) is used below
nn_parts, keep_nn_parts = [], []
for fi, f in enumerate(tif_files):
    locs_f = detections.loc[detections.file == f.name,
                            ['pos_z', 'pos_y', 'pos_x']].to_numpy()
    nn_parts.append(nearest_neighbor_distances(locs_f, psz=psz_per_file[fi]))
    keep_nn_parts.append(filter_close_beads(locs_f, psz=psz_per_file[fi],
                                            min_distance=min_nn_distance_nm, quantile=nn_quantile))
nn_nm = np.concatenate(nn_parts)
keep_nn = np.concatenate(keep_nn_parts)
fwhm_xy_px = np.nanmean(m['fwhm_px'][:, 1:], axis=1)
keep_mad = filter_mad_outliers(fwhm_xy_px, n_mad=n_mad, side=mad_side)
keep = used & keep_nn & keep_mad

detections['nn_distance_nm'] = nn_nm
detections['fwhm_xy_px'] = fwhm_xy_px
detections['keep_nn'] = keep_nn
detections['keep_mad'] = keep_mad
detections['keep'] = keep
detections.to_csv(out_path / 'detections.csv', index=False)

print(f'beads total {len(keep)} | edge-skipped {(~used).sum()} | '
      f'close-pair rejected {(used & ~keep_nn).sum()} | '
      f'MAD rejected {(used & keep_nn & ~keep_mad).sum()} | kept {int(keep.sum())}')

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(nn_nm[used], bins=30)
axs[0].set_xlabel('nearest-neighbour distance (nm)')
axs[0].set_title('NN distances')
axs[1].hist(fwhm_xy_px[used], bins=30)
axs[1].set_xlabel('per-bead FWHM xy (px)')
axs[1].set_title('per-bead FWHM')
fig.tight_layout()

In [ ]:
# one PSF per input file, each with its own pixel size; the pooled ALL_* PSF only
# when all files share the same pixel size (averaging different samplings and
# physical crop windows is not meaningful)
stem_fi = {f.stem: fi for fi, f in enumerate(tif_files)}

per_file, per_file_n = {}, {}
for fi, f in enumerate(tif_files):
    mask = keep & (file_idx == fi)
    if mask.sum() >= 3:
        per_file[f.stem] = average_stack(stack, mask)
        per_file_n[f.stem] = int(mask.sum())
    else:
        print(f'{f.name}: only {int(mask.sum())} beads after QC - skipping per-file PSF')

if uniform_psz:
    psf_all = average_stack(stack, keep)
    psf_all_d = (distill_psf(psf_all, bead_diameter=bead_diameter_nm, psz=psz_arr,
                             num_iter=num_iter, bandlimit=bandlimit)
                 if distill else None)
else:
    psf_all, psf_all_d = None, None
    print('pixel size differs between files - no pooled ALL_* PSF (per-file PSFs only)')

if distill:
    per_file_d = {s: distill_psf(p, bead_diameter=bead_diameter_nm,
                                 psz=psz_per_file[stem_fi[s]],
                                 num_iter=num_iter, bandlimit=bandlimit)
                  for s, p in per_file.items()}
else:
    per_file_d = {}

print(f'per-file PSFs: {list(per_file)}'
      + (f'; pooled PSF from {int(keep.sum())} beads' if psf_all is not None else ''))


In [ ]:
met_all = psf_metrics(psf_all, psz_arr) if psf_all is not None else None
met_all_d = psf_metrics(psf_all_d, psz_arr) if (distill and psf_all_d is not None) else None
per_file_met = {s: psf_metrics(p, psz_per_file[stem_fi[s]]) for s, p in per_file.items()}
per_file_met_d = ({s: psf_metrics(p, psz_per_file[stem_fi[s]]) for s, p in per_file_d.items()}
                  if distill else {})

fwhm_nm = lambda met: np.asarray(met['fwhm_phys'])

if met_all is not None:
    print('=== pooled PSF (avg) ===')
    print(f"  FWHM nm (z,y,x): {np.round(fwhm_nm(met_all), 1)}  sigma px: {np.round(met_all['sigma_px'], 3)}")
    print(f"  peak offset px: {np.round(met_all['peak_offset_px'], 3)}  "
          f"sidelobe ratio: {met_all['sidelobe_ratio']:.3f}  edge level: {met_all['edge_level']:.2e}")
    if met_all_d is not None:
        print('=== pooled PSF (distilled) ===')
        print(f"  FWHM nm (z,y,x): {np.round(fwhm_nm(met_all_d), 1)}  "
              f"sidelobe ratio: {met_all_d['sidelobe_ratio']:.3f}")
else:
    print('=== no pooled PSF (pixel size differs between input files) ===')

# per-file vs per-file median: flags files whose PSF differs (e.g. different
# depletion patterns) - also the comparison to use when no pooled PSF exists
if per_file_met:
    med_fw = np.median([fwhm_nm(mm) for mm in per_file_met.values()], axis=0)
    for s, mm in per_file_met.items():
        fw = fwhm_nm(mm)
        if np.any(np.abs(fw / med_fw - 1) > fwhm_outlier_rel):
            warnings.warn(f'{s}: per-file FWHM {np.round(fw, 1)} nm deviates '
                          f'>{fwhm_outlier_rel:.0%} from per-file median {np.round(med_fw, 1)} nm')

def save_psf_report(psf, met, path, suptitle, psz, gamma=report_gamma):
    # report figure: orthogonal center cuts (top) + linear marginals with FWHM
    # markers (bottom, x axis in nm centered on the crop center). Cuts use extent
    # in nm, so anisotropic z/xy pixel sizes render at true physical scale;
    # gamma brightens dim wings/sidelobes.
    fig, axs = plt.subplots(2, 3, figsize=(13, 8))
    c = np.array(psf.shape) // 2
    cuts = [(psf[c[0]],       psz[1:],      ('y (nm)', 'x (nm)'), 'center z slice'),
            (psf[:, c[1], :], psz[[0, 2]], ('z (nm)', 'x (nm)'), 'center y slice'),
            (psf[:, :, c[2]], psz[:2],      ('z (nm)', 'y (nm)'), 'center x slice')]
    vmax = psf.max()
    for ax, (sl, p, (lab0, lab1), title) in zip(axs[0], cuts):
        v = np.clip(sl, 0, None) / vmax
        ax.imshow(v ** gamma, cmap='gray',
                  extent=[0, sl.shape[1] * p[1], sl.shape[0] * p[0], 0])
        ax.set_title(title)
        ax.set_xlabel(lab1)
        ax.set_ylabel(lab0)
    for ax_i, ax in enumerate('zyx'):
        prof = psf.sum(axis=tuple(j for j in range(3) if j != ax_i))
        pos_nm = (np.arange(len(prof)) - c[ax_i]) * psz[ax_i]
        axs[1, ax_i].plot(pos_nm, prof)
        fw_nm = met['fwhm_phys'][ax_i]
        axs[1, ax_i].axvline(-fw_nm / 2, color='r', ls='--')
        axs[1, ax_i].axvline(fw_nm / 2, color='r', ls='--')
        axs[1, ax_i].set_title(f'marginal {ax}: FWHM {fw_nm:.0f} nm')
        axs[1, ax_i].set_xlabel(f'{ax} (nm)')
    fig.suptitle(suptitle)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()
    plt.close(fig)

# one report per input file, so differing optical conditions can be compared
# at a glance; the pooled report is written as report_ALL.png only if it exists
if psf_all is not None:
    save_psf_report(psf_all, met_all, out_path / 'report_ALL.png',
                    f"pooled PSF (ALL) FWHM nm (z,y,x) = {np.round(fwhm_nm(met_all), 1)}, "
                    f"{int(keep.sum())}/{len(keep)} beads kept", psz_arr)
for s in per_file:
    save_psf_report(per_file[s], per_file_met[s], out_path / f'report_{s}.png',
                    f"{s}: FWHM nm (z,y,x) = {np.round(fwhm_nm(per_file_met[s]), 1)}, "
                    f"{per_file_n[s]} beads", psz_per_file[stem_fi[s]])
if distill:
    if psf_all_d is not None:
        save_psf_report(psf_all_d, met_all_d, out_path / 'report_ALL_distilled.png',
                        f"pooled PSF distilled (ALL) FWHM nm (z,y,x) = {np.round(fwhm_nm(met_all_d), 1)}",
                        psz_arr)
    for s in per_file_d:
        save_psf_report(per_file_d[s], per_file_met_d[s], out_path / f'report_{s}_distilled.png',
                        f"{s} distilled: FWHM nm (z,y,x) = {np.round(fwhm_nm(per_file_met_d[s]), 1)}",
                        psz_per_file[stem_fi[s]])


In [ ]:
def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(x) for x in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    return o

def save_psf(path, psf, kind, metrics, n_used, psz=psz_arr, files=None):
    save_tiff_imagej(str(path), psf.astype(np.float32), axes='ZYX', pixel_size=list(psz),
                     distance_unit='nm')
    sidecar = {
        'kind': kind,
        'psz_nm': np.asarray(psz).tolist(),
        'metrics': _jsonable(metrics),
        'bead_diameter_nm': bead_diameter_nm if kind == 'distilled' else None,
        'num_iter': num_iter if kind == 'distilled' else None,
        'n_beads_used': int(n_used),
        'n_beads_total': int(len(keep)),
        'files': files if files is not None else [f.name for f in tif_files],
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    with open(str(path) + '.json', 'w') as fh:
        json.dump(_jsonable(sidecar), fh, indent=2)

if psf_all is not None:
    save_psf(out_path / 'ALL_psf_avg.tif', psf_all, 'avg', met_all, keep.sum())
    if distill:
        save_psf(out_path / 'ALL_psf_distilled.tif', psf_all_d, 'distilled', met_all_d, keep.sum())
for s, p in per_file.items():
    fi = stem_fi[s]
    save_psf(out_path / f'{s}_psf_avg.tif', p, 'avg', per_file_met[s], per_file_n[s],
             psz=psz_per_file[fi], files=[tif_files[fi].name])
    if distill:
        save_psf(out_path / f'{s}_psf_distilled.tif', per_file_d[s], 'distilled',
                 per_file_met_d[s], per_file_n[s],
                 psz=psz_per_file[fi], files=[tif_files[fi].name])

if save_aligned_stack:
    np.savez_compressed(out_path / 'aligned_stack.npz', stack=stack, used=used,
                        keep=keep, file_idx=file_idx,
                        psz_per_file=np.asarray(psz_per_file))

print('saved to', out_path)
for p in sorted(out_path.iterdir()):
    print('  ', p.name)